# E06 · Camera model, reprojection error and Perspective-n-Point

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E06_camera_model_reprojection_error_and_perspective_.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 55 min

**Skill Hub concepts:** Pinhole camera model; Intrinsics and extrinsics; Reprojection error; Perspective-n-Point

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E06")

## Map: from a 3D point to a pixel, and back to the camera pose

$$ s\begin{bmatrix}u\\v\\1\end{bmatrix} = \underbrace{\begin{bmatrix} f_x & 0 & c_x\\ 0 & f_y & c_y\\ 0&0&1\end{bmatrix}}_{K\ \text{(intrinsics)}}
\underbrace{\big[\,R \mid t\,\big]}_{\text{extrinsics: world} \to \text{camera}} \begin{bmatrix}X\\Y\\Z\\1\end{bmatrix}$$

| Piece | Meaning | Where it comes from |
|---|---|---|
| $f_x, f_y$ | focal length in pixels | calibration (checkerboard, `cv2.calibrateCamera`) |
| $c_x, c_y$ | principal point | calibration |
| distortion $k_1, k_2, p_1, p_2, k_3$ | lens bending | calibration; undistort before using the pinhole model |
| $R, t$ | where the world (board, robot base) is relative to the camera | PnP, hand-eye calibration |

**Reprojection error:** project known 3D points with the estimated parameters and measure the pixel distance to where
they were detected; RMS below about 0.5 px is a good calibration.

**Perspective-n-Point:** given $n \ge 4$ known 3D points (a chessboard's corners, an AprilTag) and their pixels, solve
for $R, t$ (`cv2.solvePnP`, OpenCV returns a Rodrigues rotation vector). This is how Gambit can find the board.

Horizontal field of view: $\text{FOV} = 2\arctan\dfrac{W/2}{f_x}$.

In [ ]:
import numpy as np
import cv2

## Task 1: project points and measure reprojection error

In [ ]:
def project(K, R, t, X):
    """X: (N, 3) world points. Return (N, 2) pixels."""
    raise NotImplementedError  # TODO

def reprojection_rms(K, R, t, X, uv_measured):
    raise NotImplementedError  # TODO

lab.check("projection", project, reprojection_rms)

## Task 2: recover the camera pose from a chessboard
`board` = (N, 3) corner coordinates on the board plane (z = 0), `uv` = detected pixels. Return `(R, t)` using
`cv2.solvePnP` and `cv2.Rodrigues`.

In [ ]:
def board_pose(K, board, uv):
    raise NotImplementedError  # TODO

lab.check("pnp", board_pose, project)

## Checkpoint: Math
**M1 (Apply).** $f_x = 600$, $c_x = 320$, point at $(0.1, 0, 1.0)$ m in the camera frame. Pixel $u$?

**M2 (Apply).** Two points have reprojection errors of 3 px and 4 px. RMS reprojection error (3 decimals)?

**M3 (Apply).** $f_x = 600$ px, image width 640 px. Horizontal field of view in degrees (1 decimal)?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Which parameters change when you bolt the same camera to a different place on the robot?
- a) intrinsics
- b) extrinsics only; intrinsics belong to the camera and lens and stay the same
- c) both

**K2 (Evaluate).** Gambit estimates the board pose with PnP from the four outer corners. List the error sources and how to
make the pose accurate and stable.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e06.h`).
3. In `roboprep/checks/e06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()